In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/izpiz06/ragadataset/RagaDataset/report.txt
/kaggle/input/datasets/izpiz06/ragadataset/RagaDataset/directory_structure.txt
/kaggle/input/datasets/izpiz06/ragadataset/RagaDataset/Carnatic/features/f972db4d-5d16-4f9a-9841-f313e1601aaa/Aruna_Sairam/December_Season_2005/Chakkani_Raja/Chakkani_Raja.taniSegKNN
/kaggle/input/datasets/izpiz06/ragadataset/RagaDataset/Carnatic/features/f972db4d-5d16-4f9a-9841-f313e1601aaa/Aruna_Sairam/December_Season_2005/Chakkani_Raja/Chakkani_Raja.flatSegNyas
/kaggle/input/datasets/izpiz06/ragadataset/RagaDataset/Carnatic/features/f972db4d-5d16-4f9a-9841-f313e1601aaa/Aruna_Sairam/December_Season_2005/Chakkani_Raja/Chakkani_Raja.tonic
/kaggle/input/datasets/izpiz06/ragadataset/RagaDataset/Carnatic/features/f972db4d-5d16-4f9a-9841-f313e1601aaa/Aruna_Sairam/December_Season_2005/Chakkani_Raja/Chakkani_Raja.pitch
/kaggle/input/datasets/izpiz06/ragadataset/RagaDataset/Carnatic/features/f972db4d-5d16-4f9a-9841-f313e1601aaa/Aruna_Sairam/December_

## Cell 1: Environment & Directory Setup

In [2]:
import os
import sys
from pathlib import Path
import torch

print("=" * 80)
print("KAGGLE ENVIRONMENT SETUP")
print("=" * 80)

print("PyTorch Version:", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Count   :", torch.cuda.device_count())

# Dataset Root
DATASET_ROOT = Path("/kaggle/input/datasets/izpiz06/ragadataset/RagaDataset")
WORK_ROOT = Path("/kaggle/working")

# Sub-directories for Organized Outputs
ANALYSIS_OUTPUT_DIR = WORK_ROOT / "outputs" / "dataset_analysis"
ANALYSIS_FIGURES_DIR = ANALYSIS_OUTPUT_DIR / "figures"

TRAINING_OUTPUT_DIR = WORK_ROOT / "outputs" / "training"
EVAL_OUTPUT_DIR = WORK_ROOT / "outputs" / "evaluation"
EVAL_FIGURES_DIR = EVAL_OUTPUT_DIR / "figures"

# Create all folders
for folder in [ANALYSIS_OUTPUT_DIR, ANALYSIS_FIGURES_DIR, TRAINING_OUTPUT_DIR, EVAL_OUTPUT_DIR, EVAL_FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

assert DATASET_ROOT.exists(), f"Dataset not found at {DATASET_ROOT}"
print("\n[OK] Dataset found at:", DATASET_ROOT)
print("[OK] Output directory structure initialized under /kaggle/working/outputs/")

KAGGLE ENVIRONMENT SETUP
PyTorch Version: 2.10.0+cu128
CUDA Available : True
Device Count   : 2

[OK] Dataset found at: /kaggle/input/datasets/izpiz06/ragadataset/RagaDataset
[OK] Output directory structure initialized under /kaggle/working/outputs/


## Cell 2: Configuration, Constants, and Unicode Normalization

In [3]:
import re
import json
import math
import csv
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

TRADITIONS = ["Carnatic", "Hindustani"]

SELECTED_RAGAS = {
    "Carnatic": [
        "Aṭāna",
        "Bhairavi",
        "Kalyāṇi",
        "Kāpi",
        "Kāṁbhōji",
        "Kēdāragauḷa",
        "Mōhanaṁ",
        "Rītigauḷa",
        "Tōḍi",
        "Śankarābharaṇaṁ",
    ],
    "Hindustani": [
        "Bhairav",
        "Jōg",
        "Bihāg",
        "Bilāsakhānī tōḍī",
        "Darbāri",
        "Khamāj",
        "Dēś",
        "Miyān malhār",
        "Yaman kalyāṇ",
        "Śrī",
    ],
}

# Preprocessing & Model Constants
K = 5                      # frequency levels per semitone
SEQ_LEN = 5000             # subsequence length
N_WINDOWS = 200            # random subsequences per recording
SEED = 42                  # random seed for reproducibility
VOCAB_SIZE = 256
TRAIN_PER_RAGA = {
    "Carnatic": 9,
    "Hindustani": 9,
}

def normalize_text(value):
    """Normalize strings for matching names while preserving originals."""
    if value is None:
        return ""

    value = str(value)
    replacements = {
        "ā": "a", "Ā": "a", "ī": "i", "Ī": "i", "ū": "u", "Ū": "u",
        "ē": "e", "Ē": "e", "ō": "o", "Ō": "o", "ṁ": "m", "ṃ": "m",
        "ṅ": "n", "ñ": "n", "ṇ": "n", "ṭ": "t", "ḍ": "d", "ṛ": "r",
        "ś": "s", "Ś": "s", "ṣ": "s", "ḷ": "l", "’": "'", "–": "-", "—": "-",
    }

    for a, b in replacements.items():
        value = value.replace(a, b)

    value = value.lower()
    value = re.sub(r"\s+", " ", value)
    return value.strip()

print("[OK] Shared configuration and normalization functions loaded.")

[OK] Shared configuration and normalization functions loaded.


## Cell 3: Dataset Deep Analysis

In [4]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_raga_mapping(tradition_dir):
    info_dir = tradition_dir / "_info_"
    mapping_path = info_dir / "ragaId_to_ragaName_mapping.json"
    if not mapping_path.exists():
        raise FileNotFoundError(f"Could not find: {mapping_path}")
    mapping = load_json(mapping_path)
    result = {}
    if isinstance(mapping, dict):
        for key, value in mapping.items():
            if isinstance(value, str):
                result[str(key)] = value
            elif isinstance(value, dict):
                name = value.get("name") or value.get("raga") or value.get("ragaName")
                if name:
                    result[str(key)] = str(name)
    return result

def load_path_mapping(tradition_dir):
    info_dir = tradition_dir / "_info_"
    mapping_path = info_dir / "path_mbid_ragaid.json"
    if not mapping_path.exists():
        raise FileNotFoundError(f"Could not find: {mapping_path}")
    return load_json(mapping_path)

def extract_records_from_mapping(mapping):
    records = []
    if isinstance(mapping, dict):
        for key, value in mapping.items():
            if isinstance(value, (list, tuple)):
                if len(value) >= 2:
                    records.append({
                        "path": str(key),
                        "mbid": str(value[0]),
                        "raga_id": str(value[1]),
                    })
            elif isinstance(value, dict):
                path = value.get("path") or value.get("filepath") or value.get("file") or key
                mbid = value.get("mbid") or value.get("MBID") or value.get("musicbrainz_id") or ""
                raga_id = value.get("ragaid") or value.get("raga_id") or value.get("ragaId") or value.get("raga")
                if raga_id is not None:
                    records.append({
                        "path": str(path),
                        "mbid": str(mbid),
                        "raga_id": str(raga_id),
                    })
    elif isinstance(mapping, list):
        for item in mapping:
            if isinstance(item, dict):
                path = item.get("path") or item.get("filepath") or item.get("file") or ""
                mbid = item.get("mbid") or item.get("MBID") or ""
                raga_id = item.get("ragaid") or item.get("raga_id") or item.get("ragaId") or item.get("raga")
                if raga_id is not None:
                    records.append({
                        "path": str(path),
                        "mbid": str(mbid),
                        "raga_id": str(raga_id),
                    })
    return records

def discover_feature_files(feature_dir):
    files = []
    if not feature_dir.exists():
        return files
    for path in feature_dir.rglob("*"):
        if path.is_file():
            files.append(path)
    return files

def identify_feature_type(path):
    name = path.name
    if name.endswith(".pitchSilIntrpPP"):
        return "pitch_post_processed"
    if name.endswith(".pitch"):
        return "pitch"
    if name.endswith(".tonicFine"):
        return "tonic_fine"
    if name.endswith(".tonic"):
        return "tonic"
    if name.endswith(".flatSegNyas"):
        return "nyas_segments"
    if name.endswith(".taniSegKNN"):
        return "tani_segments"
    return path.suffix.lower().lstrip(".") or "unknown"

def read_pitch_file_analysis(path):
    try:
        arr = np.loadtxt(path)
    except Exception:
        try:
            arr = np.genfromtxt(path, delimiter=",", comments="#")
        except Exception:
            return np.array([]), np.array([])
    if arr.size == 0:
        return np.array([]), np.array([])
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    if arr.shape[1] >= 2:
        times = arr[:, 0]
        pitches = arr[:, 1]
    else:
        pitches = arr[:, 0]
        times = np.arange(len(pitches)) * 0.005
    times = np.asarray(times, dtype=float)
    pitches = np.asarray(pitches, dtype=float)
    valid = np.isfinite(times) & np.isfinite(pitches)
    return times[valid], pitches[valid]

def read_tonic_file(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            text = f.read().strip()
        match = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", text)
        if match:
            return float(match.group())
    except Exception:
        pass
    return np.nan

def analyze_pitch(path):
    result = {
        "pitch_file": str(path),
        "pitch_rows": 0,
        "valid_pitch_frames": 0,
        "invalid_pitch_frames": 0,
        "duration_seconds": np.nan,
        "pitch_min": np.nan,
        "pitch_max": np.nan,
        "pitch_mean": np.nan,
        "pitch_median": np.nan,
        "pitch_std": np.nan,
        "voiced_ratio": np.nan,
    }
    times, pitches = read_pitch_file_analysis(path)
    result["pitch_rows"] = len(pitches)
    if len(pitches) == 0:
        return result
    valid_pitch = pitches[np.isfinite(pitches) & (pitches > 0)]
    result["valid_pitch_frames"] = len(valid_pitch)
    result["invalid_pitch_frames"] = len(pitches) - len(valid_pitch)
    if len(times) > 0:
        result["duration_seconds"] = np.nanmax(times) - np.nanmin(times)
    if len(valid_pitch) > 0:
        result["pitch_min"] = np.min(valid_pitch)
        result["pitch_max"] = np.max(valid_pitch)
        result["pitch_mean"] = np.mean(valid_pitch)
        result["pitch_median"] = np.median(valid_pitch)
        result["pitch_std"] = np.std(valid_pitch)
        result["voiced_ratio"] = len(valid_pitch) / len(pitches)
    return result

def build_tradition_table(tradition):
    tradition_dir = DATASET_ROOT / tradition
    mapping = load_path_mapping(tradition_dir)
    raga_mapping = load_raga_mapping(tradition_dir)
    records = extract_records_from_mapping(mapping)
    rows = []
    feature_root = tradition_dir / "features"
    for record in records:
        raga_id = str(record["raga_id"])
        raga_name = raga_mapping.get(raga_id, raga_id)
        selected = None
        normalized_name = normalize_text(raga_name)
        for target in SELECTED_RAGAS[tradition]:
            if normalize_text(target) == normalized_name:
                selected = target
                break
        if selected is None:
            continue
        raga_feature_dir = feature_root / raga_id
        if not raga_feature_dir.exists():
            candidates = [p for p in feature_root.iterdir() if p.is_dir() and p.name == raga_id]
            if candidates:
                raga_feature_dir = candidates[0]
        rows.append({
            "tradition": tradition,
            "raga": selected,
            "raga_id": raga_id,
            "mbid": record["mbid"],
            "metadata_path": record["path"],
            "feature_dir": str(raga_feature_dir),
        })
    return pd.DataFrame(rows)

def attach_features(df):
    all_rows = []
    for _, row in df.iterrows():
        feature_dir = Path(row["feature_dir"])
        feature_files = discover_feature_files(feature_dir)
        groups = {}
        for path in feature_files:
            feature_type = identify_feature_type(path)
            stem = path.name
            for suffix in [".pitchSilIntrpPP", ".flatSegNyas", ".taniSegKNN", ".tonicFine", ".pitch", ".tonic"]:
                if stem.endswith(suffix):
                    stem = stem[:-len(suffix)]
                    break
            groups.setdefault(stem, {})
            groups[stem][feature_type] = path
        if not groups:
            all_rows.append({
                **row.to_dict(),
                "feature_record": "", "pitch_path": "",
                "pitch_post_processed_path": "", "tonic_path": "", "tonic_fine_path": "",
            })
            continue
        for feature_record, features in groups.items():
            all_rows.append({
                **row.to_dict(),
                "feature_record": feature_record,
                "pitch_path": str(features.get("pitch", "")),
                "pitch_post_processed_path": str(features.get("pitch_post_processed", "")),
                "tonic_path": str(features.get("tonic", "")),
                "tonic_fine_path": str(features.get("tonic_fine", "")),
            })
    return pd.DataFrame(all_rows)

def analyze_dataset(df):
    rows = []
    for index, row in df.iterrows():
        result = row.to_dict()
        pitch_path = row.get("pitch_post_processed_path", "") or row.get("pitch_path", "")
        if pitch_path and Path(pitch_path).exists():
            result.update(analyze_pitch(Path(pitch_path)))
        else:
            result.update({
                "pitch_file": "", "pitch_rows": 0, "valid_pitch_frames": 0, "invalid_pitch_frames": 0,
                "duration_seconds": np.nan, "pitch_min": np.nan, "pitch_max": np.nan,
                "pitch_mean": np.nan, "pitch_median": np.nan, "pitch_std": np.nan, "voiced_ratio": np.nan,
            })
        tonic_path = row.get("tonic_fine_path", "") or row.get("tonic_path", "")
        result["tonic"] = read_tonic_file(Path(tonic_path)) if tonic_path and Path(tonic_path).exists() else np.nan
        result["has_pitch"] = bool(row.get("pitch_path") and Path(row["pitch_path"]).exists())
        result["has_pitch_post_processed"] = bool(row.get("pitch_post_processed_path") and Path(row["pitch_post_processed_path"]).exists())
        result["has_tonic"] = bool(row.get("tonic_path") and Path(row["tonic_path"]).exists())
        result["has_tonic_fine"] = bool(row.get("tonic_fine_path") and Path(row["tonic_fine_path"]).exists())
        rows.append(result)
    return pd.DataFrame(rows)

def run_deep_analysis():
    print("=" * 100)
    print("RUNNING 20 RAGA DATASET DEEP ANALYSIS")
    print("=" * 100)

    all_metadata = []
    for tradition in TRADITIONS:
        tdf = build_tradition_table(tradition)
        if not tdf.empty:
            all_metadata.append(tdf)

    if not all_metadata:
        print("ERROR: No selected ragas found.")
        return

    metadata_df = pd.concat(all_metadata, ignore_index=True)
    metadata_df.to_csv(ANALYSIS_OUTPUT_DIR / "selected_20_ragas_metadata.csv", index=False)

    feature_df = attach_features(metadata_df)
    analysis_df = analyze_dataset(feature_df)
    analysis_df.to_csv(ANALYSIS_OUTPUT_DIR / "selected_20_ragas_feature_analysis.csv", index=False)

    print(f"[OK] Analysis complete. Files saved in: {ANALYSIS_OUTPUT_DIR}")

run_deep_analysis()

RUNNING 20 RAGA DATASET DEEP ANALYSIS


KeyboardInterrupt: 

## Cell 4: Data Preprocessing & Recording-Level Dataset Builder

In [5]:
def load_raga_name_mapping(tradition):
    path = DATASET_ROOT / tradition / "_info_" / "ragaId_to_ragaName_mapping.json"
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def resolve_raga_ids(tradition, names):
    mapping = load_raga_name_mapping(tradition)
    feature_root = DATASET_ROOT / tradition / "features"
    target_norm = normalize_text(names[0])
    exact_matches = []
    norm_matches = []
    for rid, nm in mapping.items():
        if normalize_text(nm) != target_norm:
            continue
        if (feature_root / rid).exists():
            if nm == names[0]:
                exact_matches.append(rid)
            norm_matches.append(rid)

    candidates = exact_matches if exact_matches else norm_matches
    if not candidates:
        raise ValueError(
            f"Could not resolve a feature-bearing raga id for {tradition}/{names[0]}."
        )
    return candidates

def read_pitch_file(pitch_path):
    values = []
    with open(pitch_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                values.append(float(line.split("\t")[-1]))
            except (ValueError, IndexError):
                values.append(0.0)
    return np.asarray(values, dtype=np.float64)

def read_tonic(base_pitch_path):
    lookup = str(base_pitch_path)[: -len(".pitch")]
    for suffix in (".tonicFine", ".tonic"):
        path = Path(lookup + suffix)
        if path.exists():
            with open(path, "r", encoding="utf-8") as f:
                return float(f.read().strip())
    raise FileNotFoundError(f"No tonic file (.tonicFine/.tonic) for {base_pitch_path}")

def quantize_feature(pitch, tonic):
    with np.errstate(divide="ignore", invalid="ignore"):
        feature = np.round(1200.0 * np.log2(pitch / tonic) * (K / 100.0)).clip(0)
    return np.nan_to_num(feature, nan=0.0, posinf=0.0, neginf=0.0).astype(np.int32)

def sample_windows(feature, n=N_WINDOWS, length=SEQ_LEN, rng=None):
    if rng is None:
        rng = np.random
    if feature.size <= length:
        raise ValueError(f"Feature length {feature.size} <= seq_len {length}.")
    windows = np.empty((n, length), dtype=np.int32)
    for i in range(n):
        c = rng.randint(0, feature.size - length)
        windows[i] = feature[c:c + length]
    return windows

def discover_recordings(tradition, raga_id):
    feature_dir = DATASET_ROOT / tradition / "features" / raga_id
    if not feature_dir.exists():
        return []
    return sorted(feature_dir.rglob("*.pitch"))

def build_dataset(
    n_windows=N_WINDOWS,
    length=SEQ_LEN,
    train_per_raga=None,
    seed=SEED,
    verbose=True,
):
    if train_per_raga is None:
        train_per_raga = TRAIN_PER_RAGA

    rng = np.random.RandomState(seed)

    class_order = []
    for tradition in ("Carnatic", "Hindustani"):
        for name in SELECTED_RAGAS[tradition]:
            class_order.append((tradition, name))
    class_mapping = {i: name for i, (_, name) in enumerate(class_order)}

    X_train, Y_train = [], []
    X_test, Y_test = [], []
    split_info = []

    for class_idx, (tradition, name) in enumerate(class_order):
        raga_ids = resolve_raga_ids(tradition, [name])
        recordings = []
        for raga_id in raga_ids:
            recordings.extend(discover_recordings(tradition, raga_id))
        raga_id = ",".join(raga_ids)

        if not recordings:
            raise ValueError(f"No recordings for {tradition}/{name} ({raga_id})")

        recordings = list(recordings)
        rng.shuffle(recordings)

        n_train = min(train_per_raga.get(tradition, 9), len(recordings) - 1)
        train_recs = recordings[:n_train]
        test_recs = recordings[n_train:]

        for rec, is_train in ([(r, True) for r in train_recs] + [(r, False) for r in test_recs]):
            tonic = read_tonic(rec)
            feature = quantize_feature(read_pitch_file(rec), tonic)
            windows = sample_windows(feature, n=n_windows, length=length, rng=rng)
            label = np.full(windows.shape[0], class_idx, dtype=np.int64)

            if is_train:
                X_train.append(windows)
                Y_train.append(label)
                split = "train"
            else:
                X_test.append(windows)
                Y_test.append(label)
                split = "held_out"

            split_info.append({
                "tradition": tradition,
                "raga": name,
                "class_idx": class_idx,
                "raga_id": raga_id,
                "recording": str(rec),
                "split": split,
                "n_windows": windows.shape[0],
            })

        if verbose:
            print(f"[{tradition:11s}] {name:24s} train={len(train_recs)} held_out={len(test_recs)}")

    X_train = np.concatenate(X_train) if X_train else np.empty((0, length))
    Y_train = np.concatenate(Y_train) if Y_train else np.empty(0, dtype=np.int64)
    X_test = np.concatenate(X_test) if X_test else np.empty((0, length))
    Y_test = np.concatenate(Y_test) if Y_test else np.empty(0, dtype=np.int64)

    return {
        "X_train": X_train.astype(np.int64),
        "Y_train": Y_train,
        "X_test": X_test.astype(np.int64),
        "Y_test": Y_test,
        "class_mapping": class_mapping,
        "n_classes": len(class_mapping),
        "seq_len": length,
        "n_windows": n_windows,
        "vocab_size": VOCAB_SIZE,
        "k": K,
        "split_info": split_info,
    }

print("[OK] Data pipeline functions loaded.")

[OK] Data pipeline functions loaded.


## Cell 5: DeepSRGM Model Architecture Definition

In [6]:
import torch.nn as nn

class DeepSRGM(nn.Module):
    def __init__(
        self,
        rnn="lstm",
        input_length=5000,
        embedding_size=128,
        hidden_size=768,
        num_layers=1,
        num_classes=20,
        vocab_size=256,
        drop_prob=0.3,
    ):
        super(DeepSRGM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.drop_prob = drop_prob
        self.dropout = nn.Dropout(drop_prob)

        if rnn.lower() == "lstm":
            self.rnn = nn.LSTM(
                input_size=embedding_size,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
                dropout=drop_prob if num_layers > 1 else 0.0,
            )
        elif rnn.lower() == "gru":
            self.rnn = nn.GRU(
                input_size=embedding_size,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
                dropout=drop_prob if num_layers > 1 else 0.0,
            )
        else:
            raise ValueError(f"Unsupported rnn type: {rnn}")

        self.w_omega = nn.Parameter(torch.Tensor(hidden_size, hidden_size))
        self.u_omega = nn.Parameter(torch.Tensor(hidden_size, 1))
        nn.init.uniform_(self.w_omega, -0.1, 0.1)
        nn.init.uniform_(self.u_omega, -0.1, 0.1)

        self.fc1 = nn.Linear(hidden_size, 384)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(384, num_classes)

    def attention(self, rnn_output):
        u = torch.tanh(torch.matmul(rnn_output, self.w_omega))
        att = torch.matmul(u, self.u_omega)
        att_score = torch.softmax(att, dim=1)
        scored_x = rnn_output * att_score
        context = torch.sum(scored_x, dim=1)
        return context

    def forward(self, x):
        x = self.embedding(x)
        x = self.dropout(x)
        output, _ = self.rnn(x)
        attn_output = self.attention(output)
        out = self.fc1(attn_output)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

print("[OK] DeepSRGM PyTorch model definition loaded.")

[OK] DeepSRGM PyTorch model definition loaded.


## Cell 6: Multi-GPU Model Training

In [7]:
import time
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader

def recording_accuracy(model, X, Y, n_windows, threshold, device):
    model.eval()
    n_recs = len(Y) // n_windows
    correct = 0
    with torch.no_grad():
        for r in range(n_recs):
            start = r * n_windows
            block = X[start:start + n_windows].to(device)
            out = model(block)
            preds = torch.argmax(out, axis=-1)
            labels = Y[start:start + n_windows].to(device)
            matched = float(torch.sum(preds == labels))
            if matched / len(preds) >= threshold:
                correct += 1
    return correct / n_recs if n_recs else 0.0

def window_metrics(model, X, Y, device):
    model.eval()
    correct = 0
    loss_sum = 0.0
    criterion = nn.CrossEntropyLoss()
    n = len(Y)
    with torch.no_grad():
        for i in range(0, n, 128):
            xb = X[i:i + 128].to(device)
            yb = Y[i:i + 128].to(device)
            out = model(xb)
            loss_sum += float(criterion(out, yb)) * len(yb)
            preds = torch.argmax(out, axis=-1)
            correct += int((preds == yb).sum().item())
    return correct / n if n else 0.0, (loss_sum / n if n else 0.0)

def save_plots(history, results_dir):
    ep = [h["epoch"] for h in history]
    tl = [h["train_loss"] for h in history]
    vl = [h["val_loss"] for h in history]
    va = [h["val_recording_accuracy"] for h in history]

    plt.figure()
    plt.plot(ep, tl, label="train_loss")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
    plt.title("Training Loss"); plt.grid(alpha=0.3)
    plt.savefig(results_dir / "training_loss.png", dpi=120); plt.close()

    plt.figure()
    plt.plot(ep, vl, label="val_loss", color="tab:red")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
    plt.title("Validation Loss"); plt.grid(alpha=0.3)
    plt.savefig(results_dir / "validation_loss.png", dpi=120); plt.close()

    plt.figure()
    plt.plot(ep, tl, label="train_loss")
    plt.plot(ep, vl, label="val_loss")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
    plt.title("Training vs Validation Loss"); plt.grid(alpha=0.3)
    plt.savefig(results_dir / "training_vs_validation_loss.png", dpi=120); plt.close()

    plt.figure()
    plt.plot(ep, va, label="val_recording_accuracy", color="tab:green")
    plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.ylim(0, 1)
    plt.legend(); plt.title("Validation Accuracy"); plt.grid(alpha=0.3)
    plt.savefig(results_dir / "validation_accuracy.png", dpi=120); plt.close()

def save_json_file(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def build_model_config(ds):
    return {
        "rnn": "lstm", "input_length": ds["seq_len"], "embedding_size": 128,
        "hidden_size": 768, "num_layers": 1, "num_classes": ds["n_classes"],
        "vocab_size": ds["vocab_size"], "drop_prob": 0.3,
    }

# Inside run_training() in Cell 6:
def run_training(epochs=20, lr=0.0001, batch_size=40, margin_vote=0.6, seed=SEED):
    torch.manual_seed(seed)
    np.random.seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_gpus = torch.cuda.device_count()

    print("\n[1/4] Building dataset (recording-level split)...")
    ds = build_dataset(n_windows=N_WINDOWS, length=SEQ_LEN, seed=seed)
    X_train = torch.from_numpy(ds["X_train"]).long()
    Y_train = torch.from_numpy(ds["Y_train"]).long()
    X_test = torch.from_numpy(ds["X_test"]).long()
    Y_test = torch.from_numpy(ds["Y_test"]).long()

    # Save splits & audit info in training output folder
    np.save(TRAINING_OUTPUT_DIR / "X_test.npy", ds["X_test"])
    np.save(TRAINING_OUTPUT_DIR / "Y_test.npy", ds["Y_test"])
    with open(TRAINING_OUTPUT_DIR / "split_info.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(ds["split_info"][0].keys()))
        writer.writeheader()
        writer.writerows(ds["split_info"])

    print("\n[2/4] Initializing DeepSRGM Model...")
    model_cfg = build_model_config(ds)
    raw_model = DeepSRGM(**model_cfg)

    if num_gpus > 1:
        model = nn.DataParallel(raw_model)
    else:
        model = raw_model

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=lr)
    trainset = TensorDataset(X_train, Y_train)
    trainloader = DataLoader(trainset, shuffle=True, batch_size=batch_size)

    print("\n[3/4] Training Loop Starting...")
    history = []
    best_acc = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        epoch_loss = 0.0
        train_correct = 0
        train_total = 0
        t0 = time.time()

        for i, (inputs, labels) in enumerate(trainloader, 0):
            optimizer.zero_grad()
            outputs = model(inputs.to(device))
            loss = criterion(outputs, labels.to(device))
            
            if loss.dim() > 0:
                loss = loss.mean()

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            train_correct += int((torch.argmax(outputs, axis=-1) == labels.to(device)).sum().item())
            train_total += labels.size(0)

            if (i + 1) % 15 == 0:
                print(f"  Epoch {epoch}/{epochs} | Batch {i+1}/{len(trainloader)} | Loss: {(running_loss/15):.3f}")
                epoch_loss += running_loss
                running_loss = 0.0

        train_loss = float(epoch_loss / (len(trainloader) // 15 * 15)) if epoch_loss > 0 else float(running_loss / len(trainloader))
        train_acc = train_correct / train_total if train_total else 0.0

        val_loss, val_win_acc = window_metrics(model, X_test, Y_test, device)
        val_rec_acc = recording_accuracy(model, X_test, Y_test, N_WINDOWS, margin_vote, device)

        state_dict_to_save = model.module.state_dict() if hasattr(model, "module") else model.state_dict()

        if val_rec_acc > best_acc:
            best_acc = val_rec_acc
            best_state = {k: v.clone() for k, v in state_dict_to_save.items()}

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_accuracy": train_acc,
            "val_window_accuracy": val_win_acc,
            "val_recording_accuracy": val_rec_acc,
            "learning_rate": lr,
            "epoch_time": time.time() - t0,
        })
        print(f"  Epoch {epoch} | train_loss {train_loss:.4f} | train_acc {train_acc:.3f} | val_loss {val_loss:.4f} | val_rec_acc {val_rec_acc:.3f}")

    print("\n[4/4] Saving Models & Configs...")
    final_state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()

    best_checkpoint = {
        "model_state_dict": best_state,
        "model_config": model_cfg,
        "epoch": int(max(h["epoch"] for h in history)),
        "best_val_recording_accuracy": best_acc,
        "num_classes": ds["n_classes"],
        "class_mapping": ds["class_mapping"],
        "preprocessing": {"k": ds["k"], "seq_len": ds["seq_len"], "n_windows": ds["n_windows"], "seed": seed},
    }
    final_checkpoint = dict(best_checkpoint)
    final_checkpoint["model_state_dict"] = final_state
    final_checkpoint["epoch"] = epochs

    torch.save(best_checkpoint, TRAINING_OUTPUT_DIR / "best_model.pt")
    torch.save(final_checkpoint, TRAINING_OUTPUT_DIR / "final_model.pt")

    with open(TRAINING_OUTPUT_DIR / "training_history.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
        writer.writeheader()
        writer.writerows(history)

    config = {
        "experiment": "deepSRGM_20_ragas",
        "num_classes": ds["n_classes"],
        "model": model_cfg,
        "preprocessing": {"k": ds["k"], "seq_len": ds["seq_len"], "n_windows": ds["n_windows"], "seed": seed},
        "training": {"epochs": epochs, "lr": lr, "batch_size": batch_size, "margin_vote_threshold": margin_vote},
        "dataset": {"root": str(DATASET_ROOT), "train_per_raga": TRAIN_PER_RAGA},
    }
    save_json_file(TRAINING_OUTPUT_DIR / "config.json", config)
    save_json_file(TRAINING_OUTPUT_DIR / "class_mapping.json", ds["class_mapping"])

    save_plots(history, TRAINING_OUTPUT_DIR)
    print("[OK] Training complete. Artifacts saved in:", TRAINING_OUTPUT_DIR)

run_training(epochs=20)


[1/4] Building dataset (recording-level split)...
[Carnatic   ] Aṭāna                    train=9 held_out=3
[Carnatic   ] Bhairavi                 train=9 held_out=3
[Carnatic   ] Kalyāṇi                  train=9 held_out=3
[Carnatic   ] Kāpi                     train=9 held_out=3
[Carnatic   ] Kāṁbhōji                 train=9 held_out=3
[Carnatic   ] Kēdāragauḷa              train=9 held_out=3
[Carnatic   ] Mōhanaṁ                  train=9 held_out=3
[Carnatic   ] Rītigauḷa                train=9 held_out=3
[Carnatic   ] Tōḍi                     train=9 held_out=3
[Carnatic   ] Śankarābharaṇaṁ          train=9 held_out=3
[Hindustani ] Bhairav                  train=9 held_out=1
[Hindustani ] Jōg                      train=9 held_out=1
[Hindustani ] Bihāg                    train=9 held_out=1
[Hindustani ] Bilāsakhānī tōḍī         train=9 held_out=1
[Hindustani ] Darbāri                  train=9 held_out=1
[Hindustani ] Khamāj                   train=9 held_out=1
[Hindustani ] Dēś    

## Cell 7: Multi-GPU Evaluation

In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def recording_predictions(model, X, Y, n_windows, device, class_mapping):
    model.eval()
    n_recs = len(Y) // n_windows
    rows = []
    with torch.no_grad():
        for r in range(n_recs):
            start = r * n_windows
            block = X[start:start + n_windows].to(device)
            out = model(block)
            win_preds = torch.argmax(out, axis=-1)
            true = int(Y[start].item())
            votes = torch.bincount(win_preds, minlength=len(class_mapping))
            pred = int(votes.argmax().item())
            frac = float(votes[pred]) / n_windows
            rows.append({
                "recording_idx": r,
                "true_class": true,
                "true_raga": class_mapping[str(true)],
                "pred_class": pred,
                "pred_raga": class_mapping[str(pred)],
                "votes": frac,
                "correct": pred == true,
            })
    return rows

def top3_window_accuracy(model, X, Y, device):
    model.eval()
    n = len(Y)
    hit = 0
    with torch.no_grad():
        for i in range(0, n, 128):
            xb = X[i:i + 128].to(device)
            yb = Y[i:i + 128].to(device)
            out = model(xb)
            _, top3 = torch.topk(out, k=3, dim=-1)
            hit += int((top3 == yb.unsqueeze(1)).any(dim=1).sum().item())
    return hit / n if n else 0.0

def compute_class_metrics(rows, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for row in rows:
        cm[row["true_class"], row["pred_class"]] += 1
    accuracy = float(np.trace(cm)) / max(len(rows), 1)

    per_class = []
    for c in range(n_classes):
        tp = int(cm[c, c])
        support = int(cm[c, :].sum())
        pred_count = int(cm[:, c].sum())
        precision = tp / pred_count if pred_count else 0.0
        recall = tp / support if support else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        per_class.append({
            "class": c, "precision": precision, "recall": recall,
            "f1": f1, "support": support,
        })

    considered = [pc for pc in per_class if pc["support"] > 0]
    macro = {m: float(np.mean([pc[m] for pc in considered])) for m in ("precision", "recall", "f1")}
    total = sum(pc["support"] for pc in per_class) or 1.0
    weighted = {m: float(sum(pc[m] * pc["support"] for pc in per_class) / total) for m in ("precision", "recall", "f1")}
    return accuracy, cm, per_class, macro, weighted

def plot_confusion(cm, class_mapping, result_dir, normalized=False):
    if normalized:
        sums = cm.sum(axis=1, keepdims=True)
        display = np.divide(cm, sums, out=np.zeros_like(cm, dtype=float), where=sums != 0)
        fname = "normalized_confusion_matrix.png"
        fmt = ".2f"
        title = "Normalized Confusion Matrix"
    else:
        display = cm
        fname = "confusion_matrix.png"
        fmt = "d"
        title = "Confusion Matrix"

    labels = [class_mapping[str(i)] for i in range(len(class_mapping))]
    plt.figure(figsize=(11, 9))
    plt.imshow(display, interpolation="nearest", cmap="Blues")
    plt.colorbar()
    plt.xticks(range(len(labels)), labels, rotation=90)
    plt.yticks(range(len(labels)), labels)
    thresh = display.max() / 2 if display.max() else 0.5
    for i in range(display.shape[0]):
        for j in range(display.shape[1]):
            val = display[i, j]
            if normalized or val > 0:
                plt.text(j, i, format(val, fmt), ha="center", va="center",
                         color="white" if val > thresh else "black")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title(title); plt.tight_layout()
    plt.savefig(result_dir / fname, dpi=120)
    plt.close()

def write_csv(result_dir, name, fieldnames, rows):
    with open(result_dir / name, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

# Inside run_evaluation() in Cell 7:
def run_evaluation(checkpoint_file="best_model.pt"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_gpus = torch.cuda.device_count()

    config = load_json(TRAINING_OUTPUT_DIR / "config.json")
    class_mapping = load_json(TRAINING_OUTPUT_DIR / "class_mapping.json")
    class_mapping = {str(k): v for k, v in class_mapping.items()}
    n_classes = len(class_mapping)
    n_windows = config["preprocessing"]["n_windows"]

    checkpoint = torch.load(TRAINING_OUTPUT_DIR / checkpoint_file, map_location=device)
    model_cfg = checkpoint["model_config"]

    raw_model = DeepSRGM(**model_cfg)
    raw_model.load_state_dict(checkpoint["model_state_dict"])

    if num_gpus > 1:
        model = nn.DataParallel(raw_model)
    else:
        model = raw_model

    model = model.to(device)

    X = torch.from_numpy(np.load(TRAINING_OUTPUT_DIR / "X_test.npy")).long()
    Y = torch.from_numpy(np.load(TRAINING_OUTPUT_DIR / "Y_test.npy")).long()

    rows = recording_predictions(model, X, Y, n_windows, device, class_mapping)
    accuracy, cm, per_class, macro, weighted = compute_class_metrics(rows, n_classes)
    top3 = top3_window_accuracy(model, X, Y, device)

    print("=" * 60)
    print(f"Recording accuracy : {accuracy * 100:.2f}%")
    print(f"Macro F1           : {macro['f1']:.4f}")
    print(f"Weighted F1        : {weighted['f1']:.4f}")
    print(f"Macro precision    : {macro['precision']:.4f}")
    print(f"Macro recall       : {macro['recall']:.4f}")
    print(f"Top-3 (window) acc : {top3 * 100:.2f}%")
    print("=" * 60)

    write_csv(EVAL_OUTPUT_DIR, "recording_predictions.csv",
              ["recording_idx", "true_class", "true_raga", "pred_class", "pred_raga", "votes", "correct"],
              [{**r, "correct": int(r["correct"])} for r in rows])

    report_rows = [{**pc, "raga": class_mapping[str(pc["class"])]} for pc in per_class]
    for kind, vals in (("macro avg", macro), ("weighted avg", weighted)):
        report_rows.append({"class": kind, "raga": "", "precision": vals["precision"],
                            "recall": vals["recall"], "f1": vals["f1"], "support": len(rows)})

    write_csv(EVAL_OUTPUT_DIR, "classification_report.csv", ["class", "raga", "precision", "recall", "f1", "support"], report_rows)
    write_csv(EVAL_OUTPUT_DIR, "per_raga_results.csv", ["class", "raga", "precision", "recall", "f1", "support"], report_rows)

    with open(EVAL_OUTPUT_DIR / "confusion_matrix.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["class"] + [class_mapping[str(i)] for i in range(n_classes)])
        for i in range(n_classes):
            writer.writerow([class_mapping[str(i)]] + list(cm[i]))

    # Plot confusion matrices into figures folder
    plot_confusion(cm, class_mapping, EVAL_FIGURES_DIR, normalized=False)
    plot_confusion(cm, class_mapping, EVAL_FIGURES_DIR, normalized=True)

    print("[OK] Evaluation artifacts saved in:", EVAL_OUTPUT_DIR)

run_evaluation(checkpoint_file="best_model.pt")

Recording accuracy : 95.00%
Macro F1           : 0.9562
Weighted F1        : 0.9510
Macro precision    : 0.9625
Macro recall       : 0.9667
Top-3 (window) acc : 92.42%
[OK] Evaluation artifacts saved in: /kaggle/working/outputs/evaluation


In [9]:
import shutil
from pathlib import Path

# Paths
OUTPUT_DIR = Path("/kaggle/working/outputs")
ZIP_FILE_PATH = Path("/kaggle/working/ragasense_deepsrgm_results")

# Compress the entire outputs folder
shutil.make_archive(ZIP_FILE_PATH, 'zip', OUTPUT_DIR)

print(f"[OK] All results zipped successfully!")
print(f"File created: {ZIP_FILE_PATH}.zip")

[OK] All results zipped successfully!
File created: /kaggle/working/ragasense_deepsrgm_results.zip
